# Stream candidate investigation: example

Stream candidates in Euclid DR1 south (Euclid only, no DES), each given by two points along it: stream 1 at (ra, dec) = (77, −40.5) and (70, −44.5) deg, stream 2 at (45.5, −45) and (44, −51) deg. The inputs cell sets both; the second assignment wins, so stream 2 is studied below.

The investigation is held by one object, `CandidateInvestigation`: it keeps every input and result and calls the functions in order. Each cell below runs one step, so its result can be looked at and tuned before the next one. Where a step needs more tuning than the arguments of the class, the cell calls the function directly and puts the result back in the object (e.g. `cand.regions = stream_regions(...)`). The last section does everything at once with `cand.run_all_steps(...)`.

Steps:
1. inputs;
2. frame and window;
3. catalog and matched-filter map of the loaded box;
4. frame check: move the endpoints by hand until the frame follows the stream;
5. matched-filter maps over distance moduli, in the frame;
6. width;
7. on and off regions;
8. Hess diagrams (CMDs) on, off and on − off, in stars/deg²;
9. isochrone;
10. summary;
11. all at once.

In [ ]:
%load_ext autoreload
%autoreload 2

import matplotlib.pyplot as plt
import numpy as np
import os

from streameucliddes.analysis.candidate_investigation import CandidateInvestigation, hess_error_table
from streameucliddes.analysis.cmd_match_filter import get_match_filter, isochrone_cmd_track
from streameucliddes.catalogs.euclid_catalog import default_star_cuts, get_euclid_stars
from streameucliddes.plotting.plot_candidate_investigation import (
    plot_candidate_frame_check,
    plot_candidate_hess,
    plot_candidate_map,
    plot_candidate_null_test,
    plot_candidate_overview,
    plot_phi1_profile,
    plot_phi2_profile,
    plot_width_fit_map,
    plot_photometric_errors,
)
from streameucliddes.plotting.plot_cmd import plot_hess, plot_isochrone, plot_on_off_difference_hess
from streameucliddes.plotting.plot_generic_search import plot_generic_search_residual_map
from streameucliddes.stream_frames import phi12_to_radec, phi12_window_outline, radec_to_phi12

## 1. Inputs

All the inputs in one cell: a candidate of the list by its id (`CANDIDATE_ID`, from the notebook `find_candidates`: `CandidateInvestigation.from_candidate_id` takes its endpoints and its best distance modulus), or any other one by its name and two endpoints; the isochrone (by default 13 Gyr, [Fe/H] = −1.9 in the default isochrone model: `default_isochrone_grid()[0]`), the match filter (by default `get_match_filter()`: Euclid DR1 VIS vs VIS − H, template-fit VIS, `error_multiplier` 3, red cap 0.55) and the sizes of the regions.

In [ ]:
# A candidate of the list (data/candidates/candidates_info.csv, notebook find_candidates): its id. Its endpoints
# and best dm come from the list. None: the name and endpoints below.
CANDIDATE_ID = 4

NAME = "euclid_dr1_south_candidate"

# Stream 1 or stream 2: the last pair assigned is used
# Stream 1:
ENDPOINT_1 = (77.0, -40.5)  # (ra, dec) deg
ENDPOINT_2 = (70.0, -44.5)

# Stream 2:
ENDPOINT_1 = (45.5, -45)  # (ra, dec) deg
ENDPOINT_2 = (44., -51)

CATALOG = "euclid_dr1"  # Euclid DR1 stars, Euclid bands only
STAR_SELECTION = "morphology_cuts"  # or "xgboost_classifier" (see default_star_cuts)
LOAD_MARGIN_DEG = 5.0  # stars read in the (ra, dec) box of the window plus this margin: room to move the endpoints
SKY_MAP_DM = 17.  # distance modulus of the map of the loaded box: where the candidate shows up
DISTANCE_MODULI = np.round(np.arange(15.0, 20.0 + 0.125, 0.25), 2)  # grid of the maps in the frame (section 5)
# The default match filter (DEFAULT_MATCH_FILTER: Euclid DR1 VIS vs VIS - H, template-fit VIS, its width and red cap)
MATCH_FILTER = get_match_filter()

# The isochrone: the default (13 Gyr, [Fe/H] = -1.9 in DEFAULT_ISOCHRONE_MODEL); add "age", "feh" (or "metallicity",
# Z) or "isochrone_model" here to change it
INPUTS = {
    "match_filter": MATCH_FILTER,
    "distance_moduli": DISTANCE_MODULI,
}
# phi2 half-width of the window (deg) for a wide candidate, set before the stars are loaded (section 3): room for the
# regions of the null test (section 8); the default: DEFAULT_MIN_PHI2_HALF_WIDTH_DEG
WINDOW_PHI2_HALF_WIDTHS = {11: 7.0, 3: 7.0}
if CANDIDATE_ID in WINDOW_PHI2_HALF_WIDTHS:
    INPUTS["min_phi2_half_width_deg"] = WINDOW_PHI2_HALF_WIDTHS[CANDIDATE_ID]
if CANDIDATE_ID is None:
    cand = CandidateInvestigation(NAME, ENDPOINT_1, ENDPOINT_2, **INPUTS)
else:  # from the list: the endpoints, the best dm (cand.distance_modulus) and the name "candidate_<id>"
    cand = CandidateInvestigation.from_candidate_id(CANDIDATE_ID, **INPUTS)
    #if CANDIDATE_ID == 2:
    #    cand.distance_modulus = SKY_MAP_DM
    NAME, ENDPOINT_1, ENDPOINT_2, SKY_MAP_DM = cand.name, cand.endpoint_1, cand.endpoint_2, cand.distance_modulus

print(f"{cand.name}: endpoints {cand.endpoint_1} and {cand.endpoint_2} deg, sky map at dm {SKY_MAP_DM}")
cand.isochrone 

## 2. Frame and window

Set at construction: the great-circle frame through the endpoints (`cand.stream_frame`, gala `GreatCircleICRSFrame.from_endpoints`: phi1 = 0 at their midpoint), the phi1/phi2 window around them (`cand.window`) and the (ra, dec) box of the window (`cand.radec_box`).

In [ ]:
phi1, phi2 = radec_to_phi12(
    [cand.endpoint_1[0], cand.endpoint_2[0]], [cand.endpoint_1[1], cand.endpoint_2[1]], cand.stream_frame
)
print(f"Endpoints in the frame: phi1 = {np.round(phi1, 2)} deg, |phi2| <= {np.abs(phi2).max():.0e} deg")
print(f"Window (phi1_min, phi1_max, phi2_min, phi2_max): {np.round(cand.window, 2)} deg")
(ra_min, ra_max), (dec_min, dec_max) = cand.radec_box
print(f"(ra, dec) box of the window: ra {ra_min:.1f} to {ra_max:.1f} deg, dec {dec_min:.1f} to {dec_max:.1f} deg")

## 3. Catalog and matched-filter map of the loaded box

Euclid DR1 south, read once in a box generously larger than the candidate, so that the endpoints can move without reading again. The "sky map" is the matched-filter map of this box (`generic_search_single_dm`), with the great circle of the frame. It does not depend on the frame: computed once.

In [ ]:
(ra_min, ra_max), (dec_min, dec_max) = cand.radec_box
ra_range = ((ra_min - LOAD_MARGIN_DEG) % 360.0, (ra_max + LOAD_MARGIN_DEG) % 360.0)  # ra_min > ra_max: across ra = 0
dec_range = (max(dec_min - LOAD_MARGIN_DEG, -90.0), min(dec_max + LOAD_MARGIN_DEG, 90.0))
print(f"Loaded box: ra {ra_range[0]:.1f} to {ra_range[1]:.1f} deg, dec {dec_range[0]:.1f} to {dec_range[1]:.1f} deg")

catalog = get_euclid_stars(CATALOG, cuts=default_star_cuts(CATALOG, STAR_SELECTION), ra_range=ra_range, dec_range=dec_range)

In [ ]:
cand.compute_sky_map(catalog, SKY_MAP_DM)
print(f"{cand.sky_map['n_selected']:,} stars in the match filter at dm {SKY_MAP_DM}")

phi1_circle = np.linspace(-180.0, 180.0, 721)
great_circle = phi12_to_radec(phi1_circle, np.zeros_like(phi1_circle), cand.stream_frame)
window_outline = phi12_to_radec(*phi12_window_outline(cand.window), cand.stream_frame)
fig, ax = plot_generic_search_residual_map(
    cand.sky_map, tracks=[("great circle", *great_circle), ("window", *window_outline)]
)

## 4. Frame check

The stars of the window are prepared (`cand.prepare()`), and one map is made in the frame at the distance modulus of the sky map (`cand.compute_zoom_map()`). `plot_candidate_frame_check` shows, side by side, the sky map with the great circle, the window and the endpoints, and the zoomed map in phi1/phi2.

Edit the endpoints in the next cell and run it again, until the stream lies along phi2 ≈ 0: only the stars of the window are prepared again, the sky map is kept. The endpoints must stay in the loaded box (section 3); otherwise load a larger one.

In [ ]:
cand.prepare()  # the stars of the window, from the catalog of section 3
cand.compute_zoom_map()  # at the distance modulus of the sky map
fig, axes = plot_candidate_frame_check(cand)

In [ ]:
%%time

endpoint_test1 = ENDPOINT_1
endpoint_test2 = ENDPOINT_2

#endpoint_test2 = (44., -51)

# Edit the endpoints, run again, until the stream lies along phi2 = 0
cand.set_endpoints(endpoint_test1, endpoint_test2).compute_zoom_map()
fig, axes = plot_candidate_frame_check(cand)

## 5. Matched-filter maps over distance moduli, in the frame

`cand.scan_distance_moduli()`, then a few maps with `plot_candidate_map`. Keep the best distance modulus by eye: `cand.distance_modulus = ...`.

In [ ]:
%%time

DMS_TO_SHOW = np.linspace(-1.25, 1.25, 9) + cand.distance_modulus
cand.scan_distance_moduli(distance_moduli=DMS_TO_SHOW)  # over DISTANCE_MODULI; n_workers=4 to compute the maps in parallel
print(f"{len(cand.maps)} maps, dm {min(cand.maps):g} to {max(cand.maps):g}")

In [ ]:
if CANDIDATE_ID == 11:
    phi1_range=(-4,2.5)
elif CANDIDATE_ID == 1:
    phi1_range=(-4,4.)
else:
    phi1_range=None
    
if CANDIDATE_ID in [7]:
    phi2_range=(-2,2)
elif CANDIDATE_ID == 1:
    phi2_range=(-2,2)
else:
    phi2_range=None

#vmin = 0.5
#vmax = 2


fig, axes = plt.subplots(3, 3, figsize=(18.0, 7.0), layout="constrained")
for dm, ax in zip(DMS_TO_SHOW, axes.flat):
    plot_candidate_map(cand, dm, ax=ax,phi2_range=phi2_range,phi1_range=phi1_range,)

os.makedirs(cand.figure_folder, exist_ok=True)
fig.savefig(os.path.join(cand.figure_folder, f"{cand.name}_dmscann.png"), dpi=150)

In [ ]:

fig, ax = plot_candidate_map(cand, phi2_range=phi2_range,phi1_range=phi1_range)

## 6. Width

Stars in the match filter at `cand.distance_modulus` → phi2 density profile (stars/deg², masked sky left out) → Gaussian plus a background (`fit_stream_width`, Shipp et al. 2018): linear by default (`DEFAULT_BACKGROUND_DEGREE`: a field gradient near the track), a constant or a polynomial of higher degree with `background_degree`, fitted near the track only (first guess ± 2 deg, shaded): further out the field density is not constant. Then the same stars in 2D, against the model (constant along phi1): does the stream follow the straight track, and where does the model fail?

In [ ]:
# Background of the fit: the default, linear (fit_stream.DEFAULT_BACKGROUND_DEGREE); background_degree=... to change it
cand.fit_width()  # stars in the match filter at cand.distance_modulus, phi1 between the endpoints
if CANDIDATE_ID == 8:
    cand.width_fit['sigma'] = 0.5
    print(f"WARNING: candidate {CANDIDATE_ID} width fit sigma set to 1.0 deg")
if CANDIDATE_ID == 12:
    cand.width_fit['sigma'] = cand.width_fit['sigma']*2
    print(f"WARNING: candidate {CANDIDATE_ID} width fit sigma set to 0.5 deg")
fit = cand.width_fit

print(f"phi2_centre = {fit['phi2_centre']:.3f} ± {fit['errors']['phi2_centre']:.3f} deg, "
      f"sigma = {fit['sigma']:.3f} ± {fit['errors']['sigma']:.3f} deg, chi2 / n_dof = {fit['chi2']:.0f} / {fit['n_dof']}")
fig, ax = plot_phi2_profile(cand.phi2_profile, cand.width_fit)

In [ ]:
fig, axes = plot_width_fit_map(cand.width_map, cand.width_fit)

In [ ]:
# Tuning: another phi1 range (e.g. where the stream is clearest), bin size, first guess or fitted range
# cand.fit_width(phi1_range=(-2.0, 2.0), bin_deg=0.1, sigma_guess=0.3, phi2_fit_range=(-1.5, 2.0), background_degree=1)
# fig, ax = plot_phi2_profile(cand.phi2_profile, cand.width_fit)

## 7. Regions

On region: |phi2 − phi2_centre| ≤ 1.4σ; off regions: two strips, below and above the track, at 3.5σ to 6σ, both over `phi1_range`; the off density is the mean of their two densities. `phi1_range` first by eye, then from the phi1 profile of on − off densities (`on_off_phi1_profile`). The regions on the map (`draw_stream_regions`) and their areas outside the masks (`region_area_deg2`).

In [ ]:
#shift = 5 * cand.width_fit["sigma"]
shift = 0.0  # no shift, for the final analysis
if CANDIDATE_ID == 1:
    shift_off = 0.3
else:
    shift_off = shift
cand.define_regions(on_phi2_offset_deg=shift, off_phi2_offsets_deg=(-shift_off, shift_off))
print(f"Areas outside the masks: on {cand.areas['on']:.2f} deg², off {cand.areas['off']:.2f} deg²")

In [ ]:
fig, ax = plt.subplots(layout="constrained")
ax = plot_candidate_map(cand, ax=ax, phi2_range=phi2_range, phi1_range=phi1_range)

In [ ]:
os.makedirs(cand.figure_folder, exist_ok=True)
fig.savefig(os.path.join(cand.figure_folder, f"{cand.name}_onoff.png"), dpi=150)

In [ ]:
# Photometric errors of the stars of the on region, each band against its own magnitude: VIS error per VIS bin,
# H error per H bin (median; statistic="mean" for the means), both on the same figure. Bins from 16: H is brighter.
ERROR_MAG_BINS = np.arange(16.0, 26.25, 0.25)
cand.compute_photometric_errors(mag_bins=ERROR_MAG_BINS)  # in_match_filter=True: only the match-filter stars
#fig, ax = plot_photometric_errors(cand.photometric_errors)

In [ ]:
fig, ax = plt.subplots(layout="constrained")
ax = plot_phi1_profile(cand.phi1_profile, cand.regions, ax=ax)

In [ ]:
# phi1_range from the phi1 profile above: where on - off stands above 0
#cand.define_regions(phi1_range=(-3.0, 2.5))

## 8. Hess diagrams

On and off Hess diagrams with the same bins (`hess_diagram`), in stars/deg²/bin, and their difference (`difference_hess`, Shipp et al. 2018): `plot_on_off_difference_hess` shows the on, off and on − off diagrams, and the difference smoothed by one bin (`smoothed_difference=True` by default; only the figure is smoothed). Then the difference alone (`plot_candidate_hess`, smoothed unless `smooth=False`).



The isochrone in the bands of the match filter (`isochrone_cmd_track`, the same ugali model as the match filter) over the difference Hess diagram (`plot_isochrone`), with the match-filter polygon if wanted (`plot_match_filter_polygon`). Try 2–3 (age, [Fe/H], distance modulus).

In [ ]:

#cand.age = 11.0  # Gyr
#cand.metallicity = 0.0003  # Z; or feh=... instead of metallicity

HESS_COLOR_BINS = np.arange(-1.0, 1.5 + 0.05, 0.1)  # VIS - H; coarser than the defaults: few stars per bin here
HESS_MAG_BINS = np.arange(18.0, 26.0 + 0.125, 0.25)  # VIS
cand.compute_hess(color_bins=HESS_COLOR_BINS, mag_bins=HESS_MAG_BINS)  # every star of the on and off regions; per deg²

cand.compute_isochrone()  # age, Z or [Fe/H], and distance modulus of the object

fig, axes = plot_on_off_difference_hess(cand.on_hess, cand.off_hess, cand.diff_hess, isochrone_track=cand.isochrone_track)
fig, ax = plot_candidate_hess(cand, match_filter=True)
os.makedirs(cand.figure_folder, exist_ok=True)
fig.savefig(os.path.join(cand.figure_folder, f"{cand.name}_hess.png"), dpi=150)

In [ ]:
fig, ax = plt.subplots( layout="constrained")

isochrone_track = {
    "age": cand.age,
    "metallicity": cand.isochrone["metallicity"],  # the Z used (from feh when it is set)
    "distance_modulus": cand.distance_modulus,
}

plot_hess(cand.diff_hess, ax=ax, difference=True, smoothing_bins=1.0, colorbar=False)
plot_isochrone(ax, isochrone_cmd_track(**isochrone_track, match_filter=cand.match_filter))
ax.legend(loc="upper left", fontsize="small")

### Null test

The on region and the off region on its side moved off the stream (`compute_null_test`): by (`on_n_sigma` +
`off_n_sigma[0]`) sigma by default, so that the inner edge of the shifted on region is where the off regions start
and the stream is in none of the three regions; above or below the stream, the side with room in the window (the larger
unmasked on region when both fit; `direction="above"` or `"below"` to choose). Its on − off Hess diagram should show no
isochrone, and its SNR be about 0: `plot_candidate_null_test` draws it beside the candidate's, on the same colour
scale.

In [ ]:
cand.compute_null_test(color_bins=HESS_COLOR_BINS, mag_bins=HESS_MAG_BINS)
null_test = cand.null_test
print(f"Null test: regions shifted {null_test['direction']} by {null_test['shift_deg']:.2f} deg; "
      f"SNR {null_test['snr']:.2f} ({null_test['n_excess']:.0f} ± {null_test['n_excess_error']:.0f} excess stars), "
      f"candidate SNR {cand.summary()['snr']:.2f}")
fig, axes = plot_candidate_null_test(cand)
os.makedirs(cand.figure_folder, exist_ok=True)
fig.savefig(os.path.join(cand.figure_folder, f"{cand.name}_null_test.png"), dpi=150)

In [ ]:
# The Hess diagrams and the photometric errors of the on region in one CSV file (hess_error_table). Hess columns, one
# row per (colour, magnitude) bin: color_centre, mag_centre, difference_density, on_density, off_density (on - off, on,
# off: stars/deg²/bin; off: the mean of the two off regions), area_on_deg2, area_off_deg2 (the same on every row):
# density x area = stars expected in an area. Error columns beside, one row per magnitude bin of the errors (those of
# cand.compute_photometric_errors above, section 7), each band against its own magnitude: error_mag_centre_VIS and
# median_error_VIS, error_mag_centre_H and median_error_H; NaN below their last bin.
hess_table = hess_error_table(cand.on_hess, cand.off_hess, cand.photometric_errors)
os.makedirs(cand.figure_folder, exist_ok=True)
csv_path = os.path.join(cand.figure_folder, f"{cand.name}_diff_hess_errors.csv")
hess_table.to_csv(csv_path, index=False)
print(f"{len(hess_table)} rows written to {csv_path}")
hess_table.head()

## 10. Summary

`cand.summary()` and `plot_candidate_overview(cand)`.

In [ ]:
for key, value in cand.summary().items():
    print(f"{key:>15}: {value}")
fig, axes = plot_candidate_overview(cand)

## 11. All at once

A new `CandidateInvestigation` with the inputs chosen above (endpoints, distance modulus, isochrone) and `cand_all.run_all_steps(catalog, step_kwargs=...)`: the arguments given above to a step go in `step_kwargs`, by method name. It should give the same results as above: the summary lists the values that differ. It computes the sky map and every map of `DISTANCE_MODULI` again, so it takes a few minutes.

In [ ]:
%%time
cand_all = CandidateInvestigation(
    NAME, endpoint_test1, endpoint_test2,
    age=cand.age, metallicity=cand.metallicity, feh=cand.feh,
    match_filter=MATCH_FILTER,
    distance_moduli=DISTANCE_MODULI,
    distance_modulus=cand.distance_modulus,
)
cand_all.run_all_steps(catalog, step_kwargs={
    "define_regions": {"on_phi2_offset_deg": shift, "off_phi2_offsets_deg": (0.0, shift)},
    "compute_hess": {"color_bins": HESS_COLOR_BINS, "mag_bins": HESS_MAG_BINS},
})

step_by_step = cand.summary()
for key, value in cand_all.summary().items():
    print(f"{key:>15}: {value}" + ("" if value == step_by_step[key] else f"   (step by step: {step_by_step[key]})"))
fig, axes = plot_candidate_overview(cand_all)